In [ ]:
# ============================================================
# EXPERIMENT 11A: RANDOM FOREST CLASSIFIER FROM SCRATCH
# Synthetic Binary Classification Dataset (Student Performance)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1 & 2: Create Dataset and Train-Test Split
X = np.array([[1, 55], [1.5, 58], [2, 60], [2.5, 63], [3, 65], [3.5, 67],
              [4, 70], [4.5, 72], [5, 75], [5.5, 78], [6, 80], [6.5, 82],
              [7, 85], [7.5, 88], [8, 90], [8.5, 92]], dtype=float)
y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]
print(f"Training Samples: {len(X_train)}\nTesting Samples: {len(X_test)}")

# Step 3, 4, 5 & 6: Gini Impurity, Splitting, and Best Split Functions
def gini_impurity(y_in):
    if len(y_in) == 0: return 0
    _, counts = np.unique(y_in, return_counts=True)
    return 1 - np.sum((counts / len(y_in))**2)

def weighted_gini(y_l, y_r):
    n = len(y_l) + len(y_r)
    return (len(y_l)/n)*gini_impurity(y_l) + (len(y_r)/n)*gini_impurity(y_r)

def split_dataset(X_in, y_in, feat, thr):
    mask = X_in[:, feat] <= thr
    return X_in[mask], y_in[mask], X_in[~mask], y_in[~mask]

def best_split(X_in, y_in, feat_idx):
    best_g, best_f, best_t = float("inf"), None, None
    for f in feat_idx:
        for t in np.unique(X_in[:, f]):
            X_l, y_l, X_r, y_r = split_dataset(X_in, y_in, f, t)
            if len(y_l) == 0 or len(y_r) == 0: continue
            g = weighted_gini(y_l, y_r)
            if g < best_g: best_g, best_f, best_t = g, f, t
    return best_f, best_t, best_g

# Step 7 & 8: Decision Tree Node and Classifier
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature, self.threshold, self.left, self.right, self.value = feature, threshold, left, right, value

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, max_features=None):
        self.max_depth, self.min_samples_split, self.max_features = max_depth, min_samples_split, max_features
        self.root, self.feature_importance = None, None

    def fit(self, X_in, y_in):
        self.feature_importance = np.zeros(X_in.shape[1])
        self.root = self._grow_tree(X_in, y_in, 0)

    def _grow_tree(self, X_in, y_in, depth):
        n_samples, n_classes = len(y_in), len(np.unique(y_in))
        if depth >= self.max_depth or n_classes == 1 or n_samples < self.min_samples_split:
            return Node(value=self._majority_class(y_in))
        
        n_feat = X_in.shape[1]
        n_sel = int(np.sqrt(n_feat)) if self.max_features is None else self.max_features
        feat_idx = np.random.choice(n_feat, max(1, min(n_sel, n_feat)), replace=False)
        
        feat, thr, _ = best_split(X_in, y_in, feat_idx)
        if feat is None: return Node(value=self._majority_class(y_in))
        
        X_l, y_l, X_r, y_r = split_dataset(X_in, y_in, feat, thr)
        parent_g = gini_impurity(y_in)
        child_g = ((len(y_l)/n_samples)*gini_impurity(y_l) + (len(y_r)/n_samples)*gini_impurity(y_r))
        self.feature_importance[feat] += (parent_g - child_g) * n_samples
        
        return Node(feat, thr, self._grow_tree(X_l, y_l, depth+1), self._grow_tree(X_r, y_r, depth+1))

    def _majority_class(self, y_in):
        classes, counts = np.unique(y_in, return_counts=True)
        return classes[np.argmax(counts)]

    def _predict_one(self, x, node):
        if node.value is not None: return node.value
        return self._predict_one(x, node.left if x[node.feature] <= node.threshold else node.right)

    def predict(self, X_in):
        return np.array([self._predict_one(x, self.root) for x in X_in])

# Step 9: Random Forest Classifier
class RandomForest:
    def __init__(self, n_trees=10, max_depth=5, min_samples_split=2):
        self.n_trees, self.max_depth, self.min_samples_split = n_trees, max_depth, min_samples_split
        self.trees = []

    def bootstrap_sample(self, X_in, y_in):
        idx = np.random.choice(len(X_in), len(X_in), replace=True)
        return X_in[idx], y_in[idx]

    def fit(self, X_in, y_in):
        self.trees = []
        for i in range(self.n_trees):
            tree = DecisionTree(self.max_depth, self.min_samples_split)
            tree.fit(*self.bootstrap_sample(X_in, y_in))
            self.trees.append(tree)
            print(f"Tree {i + 1} trained")

    def predict(self, X_in):
        preds = np.array([tree.predict(X_in) for tree in self.trees])
        return np.array([np.unique(p, return_counts=True)[0][np.argmax(np.unique(p, return_counts=True)[1])] for p in preds.T])

# Step 10-12: Create, Train, and Predict
np.random.seed(42)
forest = RandomForest(n_trees=10, max_depth=5, min_samples_split=2)
print("\n===================================\nTRAINING RANDOM FOREST\n===================================")
forest.fit(X_train, y_train)
y_pred = forest.predict(X_test)

# Step 13 & 14: Display Predictions and Accuracy
print("\nActual vs Predicted\n==============================\nActual\tPredicted")
for yt, yp in zip(y_test, y_pred):
    print(f"{yt}\t{yp}")

acc = np.mean(y_pred == y_test)
print(f"\n===================================\nRANDOM FOREST PERFORMANCE\n===================================")
print(f"Correct Predictions: {np.sum(y_pred == y_test)}\nTotal Predictions: {len(y_test)}\nAccuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}")

# Step 15-18: Confusion Matrix and Metrics
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

print(f"\nConfusion Matrix\n-------------------------\nPredicted\n  0   1\nActual 0 {TN:3d} {FP:3d}\nActual 1 {FN:3d} {TP:3d}")

prec = TP / (TP + FP) if (TP + FP) else 0
rec = TP / (TP + FN) if (TP + FN) else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
print(f"\nClassification Metrics\nPrecision: {round(prec, 4)}\nRecall: {round(rec, 4)}\nF1 Score: {round(f1, 4)}")

# Step 19: Predict New Student
new_student = np.array([[5.0, 76]])
new_pred = forest.predict(new_student)[0]
print(f"\n===================================\nNEW STUDENT PREDICTION\n===================================")
print(f"Study Hours: {new_student[0, 0]}\nAttendance: {new_student[0, 1]}\nPredicted Class: {new_pred}\nResult: {'PASS' if new_pred == 1 else 'FAIL'}")

# Step 20: Plot Dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Fail", marker="o")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Pass", marker="x")
plt.scatter(new_student[0, 0], new_student[0, 1], marker="*", s=180, label="New Student")
plt.xlabel("Study Hours"), plt.ylabel("Attendance (%)"), plt.title("Random Forest Classification")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 11B: RANDOM FOREST CLASSIFIER USING SCIKIT-LEARN
# Breast Cancer Dataset (Real-World Dataset)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names, target_names = cancer.feature_names, cancer.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
print("==========================================\nBREAST CANCER DATASET\n==========================================")
print(f"\nDataset Shape:\n{df.shape}\n\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{list(feature_names)}\n\nTarget Classes:\n{target_names}\n\nClass Distribution:\n{df['Target'].value_counts()}")

# Step 4-5: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-7: Create and Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=None, min_samples_split=2,
                                  min_samples_leaf=1, max_features="sqrt", bootstrap=True,
                                  random_state=42, n_jobs=-1).fit(X_train, y_train)
print("\nRandom Forest Model Training Completed Successfully.")

# Step 8: Display Model Information
print("\n==========================================\nRANDOM FOREST MODEL INFORMATION\n==========================================")
print(f"Number of Trees:\n{rf_model.n_estimators}\n\nNumber of Features:\n{rf_model.n_features_in_}\n\nClasses:\n{rf_model.classes_}")

# Step 9-11: Predictions, Probabilities, and Actual vs Predicted
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred, "Probability_Malignant": y_prob[:, 0], "Probability_Benign": y_prob[:, 1]})
print(f"\n==========================================\nACTUAL VS PREDICTED\n==========================================\n{results.head(20)}")

# Steps 12-16: Calculate and Display Performance Metrics
acc, prec = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, zero_division=0)
rec, f1 = recall_score(y_test, y_pred, zero_division=0), f1_score(y_test, y_pred, zero_division=0)
print("\n==========================================\nRANDOM FOREST CLASSIFICATION PERFORMANCE\n==========================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}\nPrecision: {round(prec, 4)}\nRecall: {round(rec, 4)}\nF1 Score: {round(f1, 4)}")

# Steps 17-18: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\n==========================================\nCONFUSION MATRIX\n==========================================\n{cm}")
print(f"\n==========================================\nCLASSIFICATION REPORT\n==========================================\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Step 19: Feature Importance
feature_importance = pd.DataFrame({"Feature": feature_names, "Importance": rf_model.feature_importances_}).sort_values(by="Importance", ascending=False)
top_features = feature_importance.head(15)
print(f"\n==========================================\nTOP FEATURE IMPORTANCE\n==========================================\n{top_features}")

# Step 20: Plot Confusion Matrix
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("Random Forest Confusion Matrix"), plt.colorbar()
ticks = np.arange(len(target_names))
plt.xticks(ticks, target_names), plt.yticks(ticks, target_names)
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout(), plt.show()

# Step 21: Plot Feature Importance
plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance"), plt.ylabel("Features"), plt.title("Top 15 Important Features - Random Forest")
plt.gca().invert_yaxis(), plt.tight_layout(), plt.show()

# Step 22: Actual vs Predicted Visualization
n_samples = min(30, len(y_test))
idx = np.arange(n_samples)
plt.figure(figsize=(10, 5))
plt.scatter(idx, y_test[:n_samples], marker="o", label="Actual")
plt.scatter(idx, y_pred[:n_samples], marker="x", label="Predicted")
plt.xlabel("Test Sample Index"), plt.ylabel("Class"), plt.title("Random Forest: Actual vs Predicted")
plt.yticks([0, 1], target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 23: Display Individual Tree Depths
tree_depths = [t.get_depth() for t in rf_model.estimators_]
print("\n==========================================\nINDIVIDUAL TREE INFORMATION\n==========================================")
print(f"Tree Depths: {tree_depths}\nAverage Tree Depth: {round(np.mean(tree_depths), 2)}")

# Step 24: Predict a New Sample
new_sample = X_test[0].reshape(1, -1)
new_pred, new_prob = rf_model.predict(new_sample), rf_model.predict_proba(new_sample)
print("\n==========================================\nNEW SAMPLE PREDICTION\n==========================================")
print(f"Predicted Class: {target_names[new_pred[0]]}\n\nPrediction Probabilities:")
for i in range(len(target_names)):
    print(f"{target_names[i]}: {new_prob[0][i]:.4f}")

# Step 25: Final Result
print("\n==========================================\nFINAL RESULT\n==========================================")
print("Random Forest was successfully implemented using the Breast Cancer dataset available directly from Scikit-learn.")